# Meyer–Wallach Entanglement on IQM Spark (Analysis Report)

**Authors:** Koło Naukowe Axion

This notebook is an **analysis report** for Meyer–Wallach (MW) entanglement measurements on IQM Spark.
Hardware execution is performed by the terminal script:

```bash
python scripts/run_iqm_meyer_wallach.py --run-id <run_id>
```

Outputs are written under `evaluation_and_comparison/iqm_spark/iqm_mw_outputs/<run_id>/` as
`iqm_mw_results.csv`, `iqm_mw_scores.csv`, and `run_manifest.json`.


## Methodology

**Meyer–Wallach entanglement** quantifies global entanglement in an $n$-qubit pure state $|\psi\rangle$:

$$Q(|\psi\rangle) = \frac{2}{n} \sum_{i=1}^{n} \left(1 - \mathrm{Tr}(\rho_i^2)\right)$$

where $\rho_i$ is the reduced density matrix of qubit $i$. Values lie in $[0,1]$; higher means more entanglement.

**Hardware estimation:** For each random ansatz parameter sample we measure all qubits in the $Z$, $X$, and $Y$ bases
(3 circuits per sample). Per-qubit Bloch expectations $\langle X_i\rangle, \langle Y_i\rangle, \langle Z_i\rangle$
are estimated from shot counts, then converted to a hardware MW score via single-qubit purity from Bloch vectors.

**Comparison:** We sweep ansatz (ODRA vs simulator-optimized ring) and circuit depth, reporting mean MW with standard error
across parameter samples.


## References

- Meyer & Wallach, *Global entanglement of multipartite mixed states*, Nuclear Physics B (2002).
- Qiskit / IQM backend documentation for circuit transpilation and batch execution.
- Project ansatz definitions in `src/qbanknote/ansatzes.py` (trimmed reverse-q0 parameterization).


## 1. Load Artifacts


In [ ]:
from pathlib import Path

import json
import matplotlib.pyplot as plt
import pandas as pd

from qbanknote.paths import ensure_importable, find_project_root

ensure_importable()

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
OUTPUT_ROOT = PROJECT_ROOT / "evaluation_and_comparison/iqm_spark/iqm_mw_outputs"

# Set to a specific run directory name, or None to use the newest run.
RUN_ID = None  # e.g. "20250616_120000"

if RUN_ID is None:
  candidates = sorted([p for p in OUTPUT_ROOT.iterdir() if p.is_dir()], key=lambda p: p.name)
  if not candidates:
    raise FileNotFoundError(f"No MW output runs found under {OUTPUT_ROOT}")
  run_dir = candidates[-1]
else:
  run_dir = OUTPUT_ROOT / RUN_ID

summary_path = run_dir / "iqm_mw_results.csv"
scores_path = run_dir / "iqm_mw_scores.csv"
manifest_path = run_dir / "run_manifest.json"

results_df = pd.read_csv(summary_path)
scores_df = pd.read_csv(scores_path)
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

print(f"Run directory: {run_dir}")
print(f"Backend: {manifest.get('backend', 'n/a')}")
results_df


## 2. Summary Table


In [ ]:
print("MW comparison (higher = more entanglement):")
print("depth | ansatz           | mw_avg   | mw_std")
print("-" * 50)
for _, row in results_df.iterrows():
    print(
        f"{int(row['depth']):>5} | {row['ansatz']:<16} | "
        f"{row['mw_avg']:.6f} | {row['mw_std']:.6f}"
    )


## 3. MW vs Depth


In [ ]:
ANSATZES = tuple(results_df["ansatz"].unique())

fig, ax = plt.subplots(figsize=(8, 4))
for ansatz_name in ANSATZES:
    sub = results_df[results_df["ansatz"] == ansatz_name].sort_values("depth")
    ax.plot(sub["depth"], sub["mw_avg"], marker="o", label=ansatz_name)
    ax.fill_between(
        sub["depth"],
        sub["mw_avg"] - sub["mw_sem"],
        sub["mw_avg"] + sub["mw_sem"],
        alpha=0.2,
    )
ax.set_xlabel("Depth")
ax.set_ylabel("Mean Meyer–Wallach score")
ax.set_title("MW entanglement vs depth on IQM Spark")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Per-Sample Distributions


In [ ]:
fig, axes = plt.subplots(1, len(ANSATZES), figsize=(5 * len(ANSATZES), 4), sharey=True)
if len(ANSATZES) == 1:
    axes = [axes]

for ax, ansatz_name in zip(axes, ANSATZES):
    sub = scores_df[scores_df["ansatz"] == ansatz_name]
    for depth, group in sub.groupby("depth"):
        ax.hist(group["mw_score"], bins=12, alpha=0.5, label=f"depth={depth}")
    ax.set_title(ansatz_name)
    ax.set_xlabel("MW score")
    ax.legend()
axes[0].set_ylabel("Count")
fig.suptitle("Per-sample MW score distributions")
plt.tight_layout()
plt.show()


## 5. Optional Bloch Component Summaries


In [ ]:
bloch_cols = [c for c in scores_df.columns if c.startswith(("x_q", "y_q", "z_q"))]
if bloch_cols:
    bloch_summary = (
        scores_df.groupby(["ansatz", "depth"])[bloch_cols]
        .agg(["mean", "std"])
        .round(4)
    )
    display(bloch_summary)
else:
    print("No per-qubit Bloch columns found in scores CSV.")
